##### ***BERT预训练数据处理***
###### 为对BERT实现预训练我们首先需要以理想的格式生成数据集，以便处理前文所说的两种预训练任务。起初，BERT模型在两个庞大的图书语料库和英语维基百科的合集上预训练，但是数据量过大不适合教学使用，另一方面现成的BERT预训练模型可能不适合医学等特定领域的应用。因此，在定制数据集上进行预训练变得越来越流行。此处使用较小的语料库WikiText-2。
###### WikiText-2数据集与前文预训练Word2vec的PTB数据集相比，WikiText-2保留了原来的标点符号，适用于NSP任务，其次保留了原来的大小写和数字，且数据量更大。

In [1]:
import os 
import torch
import random
from d2l import torch as d2l

###### 在WikiText-2数据集中，每一行表示一个段落，其中在任意标点符号前面插入空格，让标点符号独立成为一个词元，例如：例如原始文本："Hello world."，处理后变成："Hello world ."。同时，只保留至少有两句话的段落。为简单起见，仅使用句号作为分隔符来拆分句子。

In [2]:
#@save
d2l.DATA_HUB['wikitext-2'] = (
    'https://s3.amazonaws.com/research.metamind.io/wikitext/'
    'wikitext-2-v1.zip', '3c914d17d80b1459be871a5039ac23e752a53cbe')

#@save
def _read_wiki(data_dir):
    file_name = os.path.join(data_dir, 'wiki.train.tokens')
    with open(file_name, 'r') as f:
        lines = f.readlines()
    # 大写字母转换为小写字母
    paragraphs = [line.strip().lower().split(' . ')
                  for line in lines if len(line.split(' . ')) >= 2]
    random.shuffle(paragraphs)
    return paragraphs

In [3]:
# NSP任务的数据
def _get_next_sentence(sentence, next_sentence, paragraphs):
    if random.random() < 0.5:
        is_next = True
    else:
        next_sentence = random.choice(random.choice(paragraphs))
        is_next = False
    return sentence, next_sentence, is_next

In [4]:
def _get_nsp_data_from_paragraph(paragraph, paragraphs, vocab, max_len):
    nsp_data_from_paragraph = []
    for i in range(len(paragraph) - 1):
        tokens_a, tokens_b, is_next = _get_next_sentence(
            paragraph[i], paragraph[i + 1], paragraphs
            )
        # 考虑一个'<cls>'词元和两个'<sep>'词元
        if len(tokens_a) + len(tokens_b) + 3 > max_len:
            continue
        tokens, segments = d2l.get_tokens_and_segments(tokens_a, tokens_b)
        nsp_data_from_paragraph.append((tokens, segments, is_next))
    return nsp_data_from_paragraph

In [5]:
# MLM任务的数据
def _replace_mlm_tokens(tokens, candidate_pred_positions, num_mlm_preds, vocab):
    # 为掩码模型的输入创建新的词元副本，其中输入可能包含替换的'<mask>'或随机词元
    mlm_input_tokens = [token for token in tokens] # tokens的副本，后续被修改成MLM的输出
    pred_positions_and_labels = [] # 每个元素是(位置，原始词)记录哪些位置被选中以及它们原来的词。
    # 打乱后用于在掩码模型任务重获取15%的随机词元进行预测
    random.shuffle(candidate_pred_positions)
    for mlm_pred_position in candidate_pred_positions:
        if len(pred_positions_and_labels) >= num_mlm_preds:
            break
        masked_token = None
        if random.random() < 0.8:
            masked_token = '<mask>'
        else:
            if random.random() < 0.5:
                masked_token = tokens[mlm_pred_position]
            else:
                masked_token = random.choice(vocab.idx_to_token)
        mlm_input_tokens[mlm_pred_position] = masked_token
        pred_positions_and_labels.append((mlm_pred_position, tokens[mlm_pred_position]))
    return mlm_input_tokens, pred_positions_and_labels

In [6]:
def _get_mlm_data_from_tokens(tokens, vocab):
    candidate_pred_positions = []
    # tokens是一个字符串列表
    for i, token in enumerate(tokens):
        # 在MLM中不会预测特殊词元
        if token in ['<cls>', '<sep>']:
            continue
        candidate_pred_positions.append(i)
    num_mlm_preds = max(1, round(len(tokens) * 0.15)) # 四舍五入
    mlm_input_tokens, pred_positions_and_labels = _replace_mlm_tokens(
        tokens, candidate_pred_positions, num_mlm_preds, vocab)
    pred_positions_and_labels = sorted(pred_positions_and_labels, key=lambda x: x[0])
    pred_positions = [v[0] for v in pred_positions_and_labels]
    mlm_pred_labels = [v[1] for v in pred_positions_and_labels]
    return vocab[mlm_input_tokens], pred_positions, vocab[mlm_pred_labels]

###### 现在我们几乎准备好为BERT预训练定制一个Dataset类。在此之前，我们仍然需要定义辅助函数_pad_bert_inputs来将特殊的`<mask>`词元附加到输入。它的参数examples包含来自两个预训练任务的辅助函数_get_nsp_data_from_paragraph和_get_mlm_data_from_tokens的输出。


In [7]:
# 由于句子长度不一样，mask的位置数也不一样，所以需要将所有批次的样本pad到一个统一的尺寸。
def _pad_bert_inputs(examples, max_len, vocab):
    """
    examples.shape(token_ids, pred_position, mlm_pred_label_ids, segments, is_next)
    """
    max_num_mlm_preds = round(max_len * 0.15) # 先确定最长于此多少个mask位置。
    # 存储处理后的所有样本
    all_token_ids, all_segments, valid_lens = [], [], []
    all_pred_positions, all_mlm_weights, all_mlm_labels = [], [], []
    nsp_labels = [] 

    # 遍历每个样本
    for (token_ids, pred_positions, mlm_pred_label_ids, segments, is_next) in examples:
        # 将token序列pad到max_len
        all_token_ids.append(torch.tensor(token_ids + [vocab['<pad>']] * (max_len - len(token_ids)), dtype=torch.long))
        # segments同理
        all_segments.append(torch.tensor(segments + [0] * (max_len - len(segments)), dtype=torch.long))

        # valid_lens不包含'<pad>'的计数，记录真实长度
        valid_lens.append(torch.tensor(len(token_ids), dtype=torch.float32))

        # 需要预测的位置长度pad到max_num_mlm_preds，用0占位
        all_pred_positions.append(torch.tensor(pred_positions + [0] * (max_num_mlm_preds - len(pred_positions)), dtype=torch.long))

        # 填充词元的预测将通过乘以0权重在损失中过滤掉
        all_mlm_weights.append(
            torch.tensor([1.0] * len(mlm_pred_label_ids) + [0.0] * (max_num_mlm_preds - len(pred_positions)), dtype=torch.float32)
        )
        
        # 标签也补齐
        all_mlm_labels.append(torch.tensor(mlm_pred_label_ids + [0] * (max_num_mlm_preds - len(mlm_pred_label_ids)), dtype=torch.long))

        nsp_labels.append(torch.tensor(is_next, dtype=torch.long))
    return (all_token_ids, all_segments, valid_lens, all_pred_positions, all_mlm_weights, all_mlm_labels, nsp_labels)

In [8]:
# 整合代码，得到wikitext数据集
class _WikiTextDataset(torch.utils.data.Dataset):
    def __init__(self, paragraphs, max_len):
        # 输入paragraphs[i]是代表段落的句子字符串列表；
        # 而输出paragraphs[i]是代表段落的句子列表，其中每个句子都是词元列表
        paragraphs = [d2l.tokenize(
            paragraph, token='word') for paragraph in paragraphs]
        sentences = [sentence for paragraph in paragraphs for sentence in paragraph]
        self.vocab = d2l.Vocab(sentences, min_freq=5, reserved_tokens=['<pad>', '<mask>', '<cls>', '<sep>'])

        # 获取NSP任务的数据
        examples = []
        for paragraph in paragraphs:
            examples.extend(_get_nsp_data_from_paragraph(
                paragraph, paragraphs, self.vocab, max_len ))
        
        # 获取MLM任务的数据
        examples = [(_get_mlm_data_from_tokens(tokens, self.vocab) + (segments, is_next)) for tokens, segments, is_next in examples]

        # pad输入
        (self.all_token_ids, self.all_segments, self.valid_lens, self.all_pred_positions, self.all_mlm_weights, self.all_mlm_labels, self.nsp_labels) = _pad_bert_inputs(
            examples, max_len, self.vocab
        )
    
    def __getitem__(self, idx):
        return (self.all_token_ids[idx], self.all_segments[idx],
                self.valid_lens[idx], self.all_pred_positions[idx], self.all_mlm_weights[idx], self.all_mlm_labels[idx], self.nsp_labels[idx])

    def __len__(self):
        return len(self.all_token_ids)

In [ ]:
# 加载数据
# 由于书中的下载地址失效，所以请自行下载文件，然后替换代码如下：
def load_data_wiki(batch_size, max_len):
    """加载wikiText-2数据集"""
    num_workers = d2l.get_dataloader_workers()

    candidate_dirs = [
        os.path.join('..', 'data', 'wikitext-2-v1'),
        os.path.join('data', 'wikitext-2-v1')
    ]
    data_dir = None
    for candidate in candidate_dirs:
        if os.path.exists(os.path.join(candidate, 'wiki.train.tokens')):
            data_dir = candidate
            break

    if data_dir is None:
        data_dir = d2l.download_extract('wikitext-2', 'wikitext-2')

    paragraphs = _read_wiki(data_dir)
    train_set = _WikiTextDataset(paragraphs, max_len)
    train_iter = torch.utils.data.DataLoader(train_set, batch_size, shuffle=True, num_workers=num_workers)

    return train_iter, train_set.vocab

In [10]:
batch_size, max_len = 512, 64
train_iter, vocab = load_data_wiki(batch_size, max_len)

for (tokens_X, segments_X, valid_lens_x, pred_positions_X, mlm_weights_X, mlm_Y, nsp_y) in train_iter:
    print(tokens_X.shape, segments_X.shape, valid_lens_x.shape,
          pred_positions_X.shape, mlm_weights_X.shape, mlm_Y.shape, nsp_y.shape)
    break

torch.Size([512, 64]) torch.Size([512, 64]) torch.Size([512]) torch.Size([512, 10]) torch.Size([512, 10]) torch.Size([512, 10]) torch.Size([512])


In [11]:
len(vocab)

20256